# Tokens: What an AI Model Actually Reads

When you type a sentence into ChatGPT or Claude, the model does **not** see your words.
It does not see letters either. It sees **tokens**.

**The kitchen analogy.** Imagine a chef who can only cook with ingredients from one fixed pantry.
The pantry is large, about 200,000 items, but it is fixed: if something isn't in there, the chef has to
build it from smaller things that are. A language model is that chef. Its pantry is called the
**vocabulary**, and each item in it is a **token**. Every sentence you send must be broken into pantry
items before the model can do anything with it, and every reply comes out one pantry item at a time.

A token is usually a common word, a piece of a longer word, a punctuation mark, or a chunk of digits.
Roughly, one token is about three quarters of an English word.

Why should a non-programmer care? Three reasons, and we will see each one:

- **Money.** AI services charge per token, not per word.
- **Memory.** A model can only hold so many tokens "in mind" at once.
- **Strange mistakes.** Several famous AI blunders, like miscounting letters in "strawberry", happen
  purely because of how text is chopped into tokens.

This notebook has three parts:

1. **Watch a real tokenizer work** and see the surprising things it does
2. **Build a tiny tokenizer ourselves**, to see there is no magic inside
3. **Why it matters**: cost, memory, and the strawberry problem

You do not need to read the code. Each code cell has a plain-English note above it saying what it does.
Just run the cells and look at the results.

## 0. Setup

**What this cell does.** It loads a real tokenizer, the one used by GPT-4o, and defines a small
helper that draws each token as a colored box. The number printed is the size of the pantry:
how many different tokens this model knows.

In [1]:
import tiktoken
from IPython.display import HTML, display

enc = tiktoken.get_encoding("o200k_base")   # the tokenizer used by GPT-4o; ~200k vocabulary
print("Vocabulary size:", enc.n_vocab)

def show_tokens(text: str):
    """Print each token in its own colored box, plus the count."""
    ids = enc.encode(text)
    colors = ["#ffd6a5", "#caffbf", "#9bf6ff", "#bdb2ff", "#ffc6ff", "#fdffb6"]
    def label(t):
        b = enc.decode_single_token_bytes(t)
        try:
            return b.decode("utf-8")
        except UnicodeDecodeError:            # a fragment of a multi-byte character: show raw bytes
            return " ".join(f"{x:02x}" for x in b)
    boxes = "".join(
        f'<span style="background:{colors[i % len(colors)]};color:#000;padding:2px 4px;'
        f'margin:1px;border-radius:3px;font-family:monospace;white-space:pre">{label(t)}</span>'
        for i, t in enumerate(ids)
    )
    display(HTML(f"<div style='font-size:16px;line-height:2'>{boxes}"
                 f" &nbsp;<b style='color:#666'>{len(ids)} tokens</b></div>"))

Vocabulary size: 200019


## 1. Watch a real tokenizer work

**How to read the output.** Each colored box is one token. Colors mean nothing; they just separate
neighbors. The bold number at the end is how many tokens the sentence cost.

**What this cell does.** It chops an ordinary sentence into tokens.

Look for two things:

- Every common word is a single token. The model has seen "quick" and "fox" so often that each got
  its own pantry slot.
- The space before each word is **included in the token**. The token is not `quick`, it is
  `␣quick` (space-quick). Tokenizers glue the space to the front of the following word.

In [2]:
show_tokens("The quick brown fox jumps over the lazy dog.")

**What this cell does.** Same idea, but with longer, rarer words.

Words the tokenizer has seen less often do not get their own slot. Instead they are assembled from
smaller pieces, like building a rare dish from basic ingredients. "Tokenization" becomes two pieces
and "counterintuitive" becomes three. The model has to figure out the meaning of the whole word from
the pieces.

In [3]:
show_tokens("Tokenization is surprisingly counterintuitive.")

### The same word, different tokens

**What this cell does.** It tokenizes the word "hello" six ways: lowercase, capitalized, all caps,
and each of those with a space in front. The numbers printed are the **token IDs**: think of them as
the pantry shelf numbers where each token lives.

Notice every version gets a **different** number. To the model, `hello`, `Hello`, and `␣hello` are
three completely unrelated pantry items. It has had to learn, from experience, that they mean the same
thing. And `HELLO` in all caps doesn't even get one slot: it is split in two. Shouting costs more.

In [4]:
for word in ["hello", " hello", "Hello", " Hello", "HELLO", " HELLO"]:
    print(f"{word!r:12} -> {enc.encode(word)}")

'hello'      -> [24912]
' hello'     -> [40617]
'Hello'      -> [13225]
' Hello'     -> [32949]
'HELLO'      -> [111642, 2699]
' HELLO'     -> [58527, 2699]


### Numbers are not numbers

**What this cell does.** It tokenizes a few numbers.

We read "1234567890" as one quantity. The model sees it as four chunks of digits, a bit like reading
a phone number in groups. The digits of pi are split into `3` `.` `141` `592` `65`, which has
nothing to do with how the value works.

This is one reason language models are unreliable at arithmetic: they never see the number as a
number, only as chunks of digit-text that happen to sit next to each other. It is also why, in the
agent tutorial, we gave the model a calculator tool instead of trusting it to do math.

In [5]:
show_tokens("1234567890")
show_tokens("3.14159265")
show_tokens("The year 2024 had 366 days.")

### Not all languages cost the same

**What this cell does.** It takes one sentence, the same meaning in four languages, and counts how
many characters and how many tokens each version uses.

The pantry was stocked mostly from English text, so English words tend to get their own slots.
Other scripts are chopped much finer. The Japanese sentence is the **shortest** in characters but
costs more tokens than English, and the Hindi sentence costs twice as many.

Since AI services bill per token and have a fixed token budget per conversation, a Hindi speaker
literally pays more and gets less room than an English speaker for the same message. This is an
active fairness concern in the field.

In [6]:
sentences = {
    "English":  "Machine learning is a branch of artificial intelligence.",
    "Spanish":  "El aprendizaje automático es una rama de la inteligencia artificial.",
    "Japanese": "機械学習は人工知能の一分野です。",
    "Hindi":    "मशीन लर्निंग कृत्रिम बुद्धिमत्ता की एक शाखा है।",
}
for lang, s in sentences.items():
    print(f"{lang:9} {len(s):3} chars -> {len(enc.encode(s)):3} tokens")

English    56 chars ->   9 tokens
Spanish    68 chars ->  11 tokens
Japanese   16 chars ->  14 tokens
Hindi      47 chars ->  18 tokens


### Code and emoji

**What this cell does.** It tokenizes a tiny snippet of computer code, then a sentence with emoji.

Code tokenizes neatly because there is a huge amount of it on the internet. Emoji do not. Each emoji
turns into **two boxes showing pairs of letters and digits**. Those are the raw bytes.

**What is a byte?** Everything stored on a computer, including text, is ultimately a sequence of
small numbers from 0 to 255 called bytes. A common letter like "a" is one byte. A pizza emoji is
four bytes. The tokenizer's smallest possible pieces are single bytes, so when it meets something it
has no slot for, it falls back to the bytes. That fallback means it can encode **any** text ever
written, in any alphabet or emoji, without ever getting stuck on an "unknown" symbol.

In [7]:
show_tokens("def add(a, b):\n    return a + b")
show_tokens("I love pizza 🍕 and tacos 🌮!")

## 2. Build a tiny tokenizer ourselves

So where does the pantry come from? Who decides that "quick" gets its own slot but "counterintuitive"
gets three? Nobody decides. It is **learned from data** by a simple recipe called
**Byte Pair Encoding**, or BPE. The recipe:

1. Start by treating every single letter as its own token.
2. Look through all your text and find the **pair of neighboring tokens that appears most often**.
3. Glue that pair together into one new token, and add it to the pantry.
4. Repeat steps 2 and 3 until the pantry is as big as you want.

**A human analogy.** Think of how people invent abbreviations. If you write "the" thousands of times,
you'd love a single symbol for it. BPE does exactly that, automatically, always abbreviating whatever
comes up most. Frequent things get short codes; rare things stay spelled out.

**What the next cell does.** It writes that recipe in about 25 lines of code. You do not need to
read it. The one detail worth knowing: we put a special marker, written `</w>`, at the end of every
word. It is just a "this is where the word ends" sign, so the tokenizer can tell "low" at the end of
a word apart from "low" at the start of "lower".

In [8]:
from collections import Counter

def train_bpe(text: str, num_merges: int):
    words = [list(w) + ["</w>"] for w in text.split()]     # each word -> list of chars + end marker
    merges = []
    for _ in range(num_merges):
        pairs = Counter()
        for w in words:
            for a, b in zip(w, w[1:]):
                pairs[(a, b)] += 1
        if not pairs:
            break
        best = max(pairs, key=pairs.get)                   # most frequent adjacent pair
        merges.append(best)
        words = [merge(w, best) for w in words]            # apply it everywhere
    return merges

def merge(word: list, pair: tuple) -> list:
    out, i = [], 0
    while i < len(word):
        if i < len(word) - 1 and (word[i], word[i + 1]) == pair:
            out.append(word[i] + word[i + 1]); i += 2
        else:
            out.append(word[i]); i += 1
    return out

def tokenize(word: str, merges: list) -> list:
    tokens = list(word) + ["</w>"]
    for pair in merges:                                    # replay merges in learned order
        tokens = merge(tokens, pair)
    return tokens

**What this cell does.** It trains our tiny tokenizer on a three-line "corpus" (a corpus is just the
text you learn from) and asks for 12 merges. Then it prints the 12 abbreviations it invented, in the
order it invented them.

Read the list top to bottom. The first merges are the pairs of letters that appeared most in the
text. Watch `t`+`h` become `th`, then `th`+`e` become `the`. Nobody told it "the" is a word; it noticed
that those letters kept showing up together.

In [9]:
corpus = """
the lower the tower the newer the power
low lower lowest new newer newest
the tower is low and the power is new
"""

merges = train_bpe(corpus, num_merges=12)
print("Learned merges, in order:")
for i, (a, b) in enumerate(merges, 1):
    print(f"  {i:2}. {a!r} + {b!r} -> {a + b!r}")

Learned merges, in order:
   1. 'w' + 'e' -> 'we'
   2. 'we' + 'r' -> 'wer'
   3. 'wer' + '</w>' -> 'wer</w>'
   4. 't' + 'h' -> 'th'
   5. 'th' + 'e' -> 'the'
   6. 'the' + '</w>' -> 'the</w>'
   7. 'o' + 'wer</w>' -> 'ower</w>'
   8. 'n' + 'e' -> 'ne'
   9. 'w' + '</w>' -> 'w</w>'
  10. 'l' + 'o' -> 'lo'
  11. 'l' + 'ower</w>' -> 'lower</w>'
  12. 't' + 'ower</w>' -> 'tower</w>'


**What this cell does.** It uses the 12 learned abbreviations to chop four words into tokens.

Two things to point out:

- **`lower` is a single token.** It appeared often in the training text, so it earned its own slot.
- **`flower` still works, even though it never appeared in the training text.** The tokenizer has no
  slot for it, so it uses `f` plus the `lower` piece it already knows. This is the key property of
  BPE: there is no such thing as a word it cannot handle. Real tokenizers do the same thing, but
  starting from bytes instead of letters, which is why they can also handle emoji and every alphabet.

In [10]:
for word in ["lower", "newest", "power", "flower"]:     # 'flower' was never in the corpus
    print(f"{word:8} -> {tokenize(word, merges)}")

lower    -> ['lower</w>']
newest   -> ['ne', 'we', 's', 't', '</w>']
power    -> ['p', 'ower</w>']
flower   -> ['f', 'lower</w>']


## 3. Why any of this matters

### The strawberry problem

Ask a chatbot how many r's are in "strawberry" and, for years, many of them answered **two**.
The correct answer is three. People found this hilarious: how can something that writes essays
fail to count letters in one word?

**What this cell does.** It shows the word "strawberry", and then a question about it, as tokens.

In [11]:
show_tokens("strawberry")
show_tokens("How many r's are in strawberry?")

Here is the explanation. The model **never sees the letters**. It sees three pantry items:
`st`, `raw`, and `berry`. To count r's, it has to remember, from training, how many r's are hidden
inside each piece and then add them up in its head. It cannot look. It is like asking someone to count
the r's in a word when all you show them is three cards labeled with syllables.

Spelling words backwards, counting letters, and rhyming are all hard for the same reason: they are
questions about letters, and the model does not have letters.

**Bonus twist.** Look at the second output. When "strawberry" appears mid-sentence, with a space in
front of it, it is **one** token, not three. So the model literally sees a different puzzle depending
on where the word sits in the sentence.

### Cost and memory

Two facts about every AI service:

- **You pay per token.** The bill is not by word or by page. It is by token in, and token out.
- **The model has a fixed working memory, measured in tokens.** This is called the **context
  window**. Everything in the conversation so far, plus the reply, must fit inside it. Once it is
  full, the oldest material has to be dropped or summarized.

**What this cell does.** It counts the words and tokens in a paragraph, then works out roughly how
much text fits in a typical 200,000-token context window.

In [12]:
paragraph = (
    "Attention is all you need. The dominant sequence transduction models are based on complex "
    "recurrent or convolutional neural networks that include an encoder and a decoder. The best "
    "performing models also connect the encoder and decoder through an attention mechanism."
)
n_words, n_tokens = len(paragraph.split()), len(enc.encode(paragraph))
print(f"{n_words} words -> {n_tokens} tokens  ({n_tokens / n_words:.2f} tokens per word)")
print()
print("Rule of thumb for English: ~1.3 tokens per word, ~4 characters per token.")
print(f"A 200k-token context window holds roughly {200_000 / 1.3:,.0f} English words, about two long novels.")

40 words -> 45 tokens  (1.12 tokens per word)

Rule of thumb for English: ~1.3 tokens per word, ~4 characters per token.
A 200k-token context window holds roughly 153,846 English words, about two long novels.


For English, a handy rule: **about 1.3 tokens per word**, or **about 4 letters per token**.
So a 200,000-token window holds around 150,000 words, roughly two long novels. That sounds like a
lot until you paste in a long document, a spreadsheet export, or a codebase.

### Nothing is lost

One reassuring point to end on. Tokenizing is not a summary or an approximation. It is like
disassembling a LEGO model into bricks: you can always put it back together exactly.

**What this cell does.** It turns a sentence into token numbers, turns the numbers back into text,
and checks that the result is identical to the original.

In [13]:
text = "Tokens → IDs → tokens. Lossless!"
ids = enc.encode(text)
print(ids)
print(enc.decode(ids))
assert enc.decode(ids) == text

[30325, 15155, 58321, 15155, 20290, 13, 29377, 2695, 0]
Tokens → IDs → tokens. Lossless!


## Talking points for 15 minutes

| Minute | Beat | Cell |
|---|---|---|
| 0-2 | "AI models don't read words." Show a sentence in colored boxes. Point out the glued-on spaces. | Section 1 |
| 2-5 | Audience guesses: is `hello` the same as `Hello`? Numbers as digit chunks. Languages cost differently. | Section 1 |
| 5-9 | Build a tokenizer live. Show abbreviations appearing in frequency order. Tokenize `flower`, a word it never saw. | Section 2 |
| 9-12 | Strawberry. Let them see the three cards. "It cannot look at the letters." | Section 3 |
| 12-14 | You pay per token; the model remembers in tokens. Two novels. | Section 3 |
| 14-15 | One line to close: "A tokenizer is a compression scheme, and the model lives inside it." | |

Live-demo idea: ask the audience for a word or sentence and run `show_tokens` on it.
Names, foreign words, and made-up words all produce surprising splits.